# Deploy MCP Servers on OpenShift

This notebook deploys MCP servers as shared services on OpenShift, accessible by all team members via Routes.

**Servers to deploy:**

| # | Server | Type | Air-gapped | Purpose |
|---|--------|------|:----------:|---------|
| 1 | Context7 | External API | No | Library documentation lookup |
| 2 | SearXNG | Self-hosted meta-engine | No | Web search & page fetching |
| 3 | Code Sandbox | Local | Yes | Secure Python/Bash/Node execution |
| 4 | Codebase Search | Custom (AI) | Yes | Semantic code search over internal repo |
| 5 | Repo Docs | Custom (AI) | Yes | Internal documentation Q&A |

> Servers 4-5 use **sentence-transformers** for local embeddings — no external LLM dependency.
> In air-gapped environments, skip servers 1-2 (they require internet).

## 1. Verify Cluster Access

In [ ]:
%%bash
echo "Cluster: $(oc whoami --show-server)"
echo "User: $(oc whoami)"
echo ""
echo "Apps domain (for Route URLs):"
oc get ingresses.config cluster -o jsonpath='{.spec.domain}'
echo ""

## 2. Create Namespace

All MCP servers deploy into the `mcp-servers` namespace.

In [ ]:
%%bash
oc apply -f manifests/00-namespace-secret.yaml
echo ""
echo "Namespace ready:"
oc get ns mcp-servers -o jsonpath='{.metadata.name} (status: {.status.phase})'
echo ""

## 3. Server 1 — Context7 (Library Documentation)

Context7 provides up-to-date library documentation for AI agents.
Requires outbound internet to call Upstash's API.

**Tools:** `resolve-library-id`, `get-library-docs`

| Environment | Works? |
|-------------|--------|
| Internet access | Yes |
| Air-gapped | No (skip this cell) |

In [ ]:
%%bash
echo "Checking outbound internet access..."
HTTP_CODE=$(curl -s -o /dev/null -w "%{http_code}" -m 5 https://mcp.context7.com/mcp)

if [ "$HTTP_CODE" = "200" ] || [ "$HTTP_CODE" = "405" ]; then
    echo "Internet reachable - deploying Context7..."
    oc apply -f manifests/02-context7.yaml
    echo ""
    oc wait --for=condition=available deployment/mcp-context7 -n mcp-servers --timeout=120s 2>/dev/null \
        && echo "Context7 Pod ready" \
        || echo "Pod still starting... check: oc get pods -n mcp-servers"
    echo ""
    echo "Route URL:"
    oc get route mcp-context7 -n mcp-servers -o jsonpath='https://{.spec.host}/mcp'
    echo ""
else
    echo "No internet access (HTTP $HTTP_CODE) - skipping Context7."
    echo "This server requires outbound connectivity to Upstash API."
fi

## 4. Server 2 — SearXNG (Web Search)

SearXNG is a self-hosted meta-search engine bundled as a single Docker image.
No external API keys, no rate limits — searches via multiple upstream engines.

**Tools:** `search-web`, `fetch-web`

| Environment | Works? |
|-------------|--------|
| Internet access | Yes |
| Air-gapped | No (skip this cell) |

In [ ]:
%%bash
echo "Checking outbound internet access..."
HTTP_CODE=$(curl -s -o /dev/null -w "%{http_code}" -m 5 https://www.google.com)

if [ "$HTTP_CODE" = "200" ] || [ "$HTTP_CODE" = "301" ] || [ "$HTTP_CODE" = "302" ]; then
    echo "Internet reachable - deploying SearXNG..."
    oc apply -f manifests/04-searxng.yaml
    echo ""
    oc wait --for=condition=available deployment/mcp-searxng -n mcp-servers --timeout=180s 2>/dev/null \
        && echo "SearXNG Pod ready" \
        || echo "Pod still starting... check: oc get pods -n mcp-servers"
    echo ""
    echo "Route URL:"
    oc get route mcp-searxng -n mcp-servers -o jsonpath='https://{.spec.host}/mcp'
    echo ""
else
    echo "No internet access (HTTP $HTTP_CODE) - skipping SearXNG."
    echo "This server requires outbound connectivity for search results."
fi

## 5. Server 3 — Code Sandbox (Secure Execution)

Secure code execution sandbox. Runs Python, Bash, and Node.js in an isolated workspace.
No external dependencies — works fully offline.

**Tools:** `execute_code`, `read_file`, `write_file`, `list_files`

| Environment | Works? |
|-------------|--------|
| Internet access | Yes |
| Air-gapped | Yes |

In [ ]:
%%bash
echo "Deploying Code Sandbox MCP server..."
oc apply -f manifests/03-code-sandbox.yaml

echo ""
oc wait --for=condition=available deployment/mcp-code-sandbox -n mcp-servers --timeout=180s 2>/dev/null \
    && echo "Code Sandbox ready" \
    || echo "Pod still starting..."

echo ""
echo "Route URL:"
oc get route mcp-code-sandbox -n mcp-servers -o jsonpath='https://{.spec.host}/mcp'
echo ""
echo ""
echo "Health check:"
ROUTE=$(oc get route mcp-code-sandbox -n mcp-servers -o jsonpath='{.spec.host}')
curl -sk "https://${ROUTE}/health"
echo ""

## 6. Custom AI Servers — Build Images

Servers 4 and 5 use **sentence-transformers** for local semantic search.
We build container images with the embedding model pre-downloaded (avoids runtime download).

**Build time**: ~5-8 minutes (downloads `all-MiniLM-L6-v2` model, ~80MB)

In [ ]:
%%bash
echo "=== Building mcp-codebase-search image ==="
echo ""

if ! oc get bc mcp-codebase-search -n mcp-servers &>/dev/null; then
    oc new-build --binary --strategy=docker --name=mcp-codebase-search -n mcp-servers 2>&1
    echo ""
fi

echo "Starting build (this takes ~5 min)..."
oc start-build mcp-codebase-search \
    --from-dir=./mcp-codebase-search \
    -n mcp-servers \
    --follow --wait

In [ ]:
%%bash
echo "=== Building mcp-repo-docs image ==="
echo ""

if ! oc get bc mcp-repo-docs -n mcp-servers &>/dev/null; then
    oc new-build --binary --strategy=docker --name=mcp-repo-docs -n mcp-servers 2>&1
    echo ""
fi

echo "Starting build (this takes ~5 min)..."
oc start-build mcp-repo-docs \
    --from-dir=./mcp-repo-docs \
    -n mcp-servers \
    --follow --wait

In [ ]:
%%bash
echo "Build results:"
echo ""
oc get builds -n mcp-servers --sort-by=.metadata.creationTimestamp | tail -5
echo ""
echo "Image streams:"
oc get is -n mcp-servers -o custom-columns=NAME:.metadata.name,TAGS:.status.tags[0].tag,UPDATED:.status.tags[0].items[0].created

## 7. Create Data ConfigMaps

Servers 4 and 5 perform semantic search over *your* code and docs.
They need the actual files mounted into their pods — we package them as ConfigMaps.

- **cafe-source-code**: Python source files from `cafe-order-system` (indexed by Codebase Search)
- **cafe-docs**: Internal documentation — architecture, API guide, runbook, etc. (indexed by Repo Docs)

In [ ]:
%%bash
CAFE_APP_DIR="../0_setup/apps/cafe-order-system"

echo "=== Creating cafe-source-code ConfigMap ==="

oc delete configmap cafe-source-code -n mcp-servers 2>/dev/null

oc create configmap cafe-source-code -n mcp-servers \
    --from-file=main.py=${CAFE_APP_DIR}/app/main.py \
    --from-file=config.py=${CAFE_APP_DIR}/app/config.py \
    --from-file=database.py=${CAFE_APP_DIR}/app/database.py \
    --from-file=models.py=${CAFE_APP_DIR}/app/models.py \
    --from-file=schemas.py=${CAFE_APP_DIR}/app/schemas.py \
    --from-file=routes_menu.py=${CAFE_APP_DIR}/app/routes/menu.py \
    --from-file=routes_orders.py=${CAFE_APP_DIR}/app/routes/orders.py \
    --from-file=routes_customers.py=${CAFE_APP_DIR}/app/routes/customers.py \
    --from-file=services_order.py=${CAFE_APP_DIR}/app/services/order_service.py \
    --from-file=services_inventory.py=${CAFE_APP_DIR}/app/services/inventory_service.py \
    --from-file=requirements.txt=${CAFE_APP_DIR}/requirements.txt \
    --from-file=Dockerfile=${CAFE_APP_DIR}/Dockerfile

echo ""
echo "=== Creating cafe-docs ConfigMap ==="

oc delete configmap cafe-docs -n mcp-servers 2>/dev/null

oc create configmap cafe-docs -n mcp-servers \
    --from-file=architecture.md=${CAFE_APP_DIR}/docs/architecture.md \
    --from-file=api-guide.md=${CAFE_APP_DIR}/docs/api-guide.md \
    --from-file=onboarding.md=${CAFE_APP_DIR}/docs/onboarding.md \
    --from-file=security-policy.md=${CAFE_APP_DIR}/docs/security-policy.md \
    --from-file=runbook.md=${CAFE_APP_DIR}/docs/runbook.md

echo ""
echo "ConfigMaps created:"
oc get configmap -n mcp-servers --no-headers | grep cafe

## 8. Server 4 — Codebase Search (Semantic Code Search)

Semantic search over internal source code using sentence-transformers embeddings.
Indexes all Python files from the `cafe-order-system` demo app.

**Tools:** `search_code`, `get_file`, `list_files`

| Environment | Works? |
|-------------|--------|
| Internet access | Yes |
| Air-gapped | Yes (image includes pre-downloaded model) |

In [ ]:
%%bash
echo "Deploying Codebase Search MCP server..."
oc apply -f manifests/05-codebase-search.yaml

echo ""
echo "Waiting for pod (embedding model loading takes ~30s)..."
oc wait --for=condition=available deployment/mcp-codebase-search -n mcp-servers --timeout=300s 2>/dev/null \
    && echo "Codebase Search ready" \
    || echo "Pod still starting... check: oc logs deploy/mcp-codebase-search -n mcp-servers"

echo ""
echo "Route URL:"
oc get route mcp-codebase-search -n mcp-servers -o jsonpath='https://{.spec.host}/mcp'
echo ""

## 9. Server 5 — Repo Docs (Documentation Q&A)

Semantic search over internal documentation (architecture, API guides, runbooks, security policies).
Uses sentence-transformers for local embedding — no external LLM calls.

**Tools:** `search_docs`, `list_docs`

| Environment | Works? |
|-------------|--------|
| Internet access | Yes |
| Air-gapped | Yes (image includes pre-downloaded model) |

In [ ]:
%%bash
echo "Deploying Repo Docs MCP server..."
oc apply -f manifests/06-repo-docs.yaml

echo ""
echo "Waiting for pod (embedding model loading takes ~30s)..."
oc wait --for=condition=available deployment/mcp-repo-docs -n mcp-servers --timeout=300s 2>/dev/null \
    && echo "Repo Docs ready" \
    || echo "Pod still starting... check: oc logs deploy/mcp-repo-docs -n mcp-servers"

echo ""
echo "Route URL:"
oc get route mcp-repo-docs -n mcp-servers -o jsonpath='https://{.spec.host}/mcp'
echo ""

## 10. Verify All Servers

![mcp server custom](../images/mcp_servers_custom1.png)

In [1]:
%%bash
echo "MCP Server Deployment Status"
echo "============================================================"
echo ""
echo "=== Pods ==="
oc get pods -n mcp-servers --sort-by=.metadata.name

echo ""
echo "=== Routes (MCP Endpoints) ==="
echo ""
printf "%-25s %-10s %s\n" "SERVER" "AIR-GAP" "ENDPOINT"
printf "%-25s %-10s %s\n" "-------" "-------" "--------"

for route in $(oc get routes -n mcp-servers -o jsonpath='{.items[*].metadata.name}'); do
    host=$(oc get route $route -n mcp-servers -o jsonpath='{.spec.host}')
    case $route in
        mcp-context7|mcp-searxng) airgap="No" ;;
        *) airgap="Yes" ;;
    esac
    printf "%-25s %-10s %s\n" "$route" "$airgap" "https://${host}/mcp"
done

MCP Server Deployment Status

=== Pods ===
NAME                                   READY   STATUS      RESTARTS         AGE
image-debug-42d9r                      0/1     Completed   0                3d1h
mcp-code-sandbox-78d984cb56-t8fm9      1/1     Running     1                18h
mcp-codebase-search-2-build            0/1     Completed   0                3d23h
mcp-codebase-search-3-build            0/1     Completed   0                3d22h
mcp-codebase-search-4-build            0/1     Completed   0                2d22h
mcp-codebase-search-5-build            0/1     Completed   0                7h9m
mcp-codebase-search-59778887c7-zxs22   1/1     Running     4                3d22h
mcp-codebase-search-6-build            0/1     Completed   0                6h40m
mcp-context7-5497d6c7c9-nmq4g          1/1     Running     33 (2m25s ago)   5h16m
mcp-duckduckgo-5cf577c995-hj7kt        1/1     Running     3 (89m ago)      5h51m
mcp-repo-docs-1-build                  0/1     Completed   0 

In [ ]:
import subprocess, json

result = subprocess.run(
    ["oc", "get", "routes", "-n", "mcp-servers",
     "-o", "jsonpath={range .items[*]}{.metadata.name}={.spec.host}\n{end}"],
    capture_output=True, text=True
)

print("MCP Health Check (all servers):")
print("=" * 60)

init_payload = json.dumps({
    "jsonrpc": "2.0", "id": 1, "method": "initialize",
    "params": {"protocolVersion": "2025-03-26", "capabilities": {},
               "clientInfo": {"name": "healthcheck", "version": "1.0"}}
})

for line in result.stdout.strip().split("\n"):
    if "=" in line:
        name, host = line.split("=", 1)
        url = f"https://{host}/mcp"
        r = subprocess.run(
            ["curl", "-sk", "-X", "POST",
             "-H", "Content-Type: application/json",
             "-H", "Accept: application/json, text/event-stream",
             "-d", init_payload,
             "-o", "/dev/null", "-w", "%{http_code}", "-m", "10", url],
            capture_output=True, text=True)
        code = r.stdout.strip()
        status = "PASS" if code == "200" else f"FAIL ({code})"
        print(f"  [{status}] {name}: {url}")

if not result.stdout.strip():
    print("  No routes found. Deploy servers first.")

## Summary

| Server | Type | Air-gapped | Key Tools |
|--------|------|:----------:|-----------|
| Context7 | External API | No | `resolve-library-id`, `get-library-docs` |
| SearXNG | Self-hosted meta-engine | No | `search-web`, `fetch-web` |
| Code Sandbox | Local runtime | Yes | `execute_code`, `read_file`, `write_file` |
| Codebase Search | AI Embeddings | Yes | `search_code`, `get_file`, `list_files` |
| Repo Docs | AI Embeddings | Yes | `search_docs`, `list_docs` |

**Air-gapped deployment**: Code Sandbox + Codebase Search + Repo Docs all work without internet.

## Next Steps

- `3_integrate_mcp_catalog.ipynb` — Register MCP servers with RHOAI MCP Catalog
- `4_connect_ide_clients.ipynb` — Configure your IDE to connect to these MCP servers
- `../3_basic_run/` — Run the coding assistant with all tools enabled